In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted successfully")


# ============================================================
# CELL 2: Imports & Configuration
# ============================================================
import os, gc, pickle, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────
BASE_PATH        = "/content/drive/MyDrive/new obada/"
PREPROCESSED     = BASE_PATH + "data/preprocessed/"
MODEL_DIR        = BASE_PATH + "models/"
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Appliance ─────────────────────────────────────────────────
APPLIANCE        = "dishwasher"
TRAIN_PKL        = PREPROCESSED + f"{APPLIANCE}_train.pkl"
VAL_PKL          = PREPROCESSED + f"{APPLIANCE}_val.pkl"
MODEL_PATH       = MODEL_DIR    + f"{APPLIANCE}_best.pt"

# ── Hyperparameters ───────────────────────────────────────────
WINDOW_SIZE      = 599
BATCH_SIZE       = 512
LR               = 3e-4
MAX_EPOCHS       = 30
PATIENCE         = 7         # early stopping
VAL_RATIO        = 6         # 1:6  ON:OFF during training validation
RANDOM_SEED      = 42

# ── Device ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config ready | device: {device}")
print(f"   Train PKL : {TRAIN_PKL}")
print(f"   Val PKL   : {VAL_PKL}")


# ============================================================
# CELL 3: Dataset
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y_label):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)   # (N,1,599)
        self.y = torch.tensor(y_label, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def load_pkl(path):
    with open(path, "rb") as f:
        d = pickle.load(f)
    return d["X"], d["y_label"]


def make_val_sample(X_val, y_val, ratio=6, seed=42):
    """
    Use seed=42 for first display, then pass epoch number as seed.
    """
    rng     = np.random.default_rng(seed)
    on_idx  = np.where(y_val == 1)[0]
    off_idx = np.where(y_val == 0)[0]
    n_off   = min(len(on_idx) * ratio, len(off_idx))
    sampled = rng.choice(off_idx, size=n_off, replace=False)
    idx     = np.sort(np.concatenate([on_idx, sampled]))
    return X_val[idx], y_val[idx]


print("✅ Dataset helpers defined")


# ============================================================
# CELL 4: Model — CNN-BiLSTM
# ============================================================
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# ─────────────────────────────────────────────────────────────
class CNNBiLSTM(nn.Module):
    def __init__(self, window=599):
        super().__init__()

        # ── CNN Encoder ────────────────────────────────────────
        self.cnn = nn.Sequential(
            # [AR_COMMENT_REMOVED]
            nn.Conv1d(1, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 599 → 299

            # Block 2
            nn.Conv1d(32, 64, kernel_size=9, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 299 → 149

            # Block 3
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 149 → 74
        )

        # ── BiLSTM ─────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )

        # ── Classifier ─────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            # [AR_COMMENT_REMOVED]
        )

    def forward(self, x):
        # x: (B, 1, 599)
        feat = self.cnn(x)                     # (B, 128, 74)
        feat = feat.permute(0, 2, 1)           # (B, 74, 128)
        out, _ = self.lstm(feat)               # (B, 74, 128)
        # [AR_COMMENT_REMOVED]
        out = out[:, -1, :]                    # (B, 128)
        logits = self.classifier(out)          # (B, 1)
        return logits.squeeze(1)               # (B,)


model = CNNBiLSTM(WINDOW_SIZE).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ CNN-BiLSTM ready | params: {total_params:,}")
print(model)


# ============================================================
# CELL 5: Training
# ============================================================
print("\n" + "="*55)
print(" LOADING DATA")
print("="*55)

X_train, y_train = load_pkl(TRAIN_PKL)
X_val_full, y_val_full = load_pkl(VAL_PKL)

print(f"   Train: {X_train.shape} | ON: {int(y_train.sum()):,} ({100*y_train.mean():.1f}%)")
print(f"   Val  : {X_val_full.shape} | ON: {int(y_val_full.sum()):,} ({100*y_val_full.mean():.1f}%)")

# [AR_COMMENT_REMOVED]
on_idx_global   = np.where(y_val_full == 1)[0]
off_idx_global  = np.where(y_val_full == 0)[0]
n_off_per_epoch = min(len(on_idx_global) * VAL_RATIO, len(off_idx_global))
print(f"   Val per epoch → {len(on_idx_global)+n_off_per_epoch:,} windows | "
      f"ON: {len(on_idx_global):,} | OFF: {n_off_per_epoch:,} (rotating each epoch)")

# [AR_COMMENT_REMOVED]
train_ds     = NILMDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)

print(f"\n   Train batches: {len(train_loader)}")

# ── Loss & Optimizer ──────────────────────────────────────────
# [AR_COMMENT_REMOVED]
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# ── Training Loop ─────────────────────────────────────────────
best_f1       = 0.0
patience_cnt  = 0
history       = []

print("\n" + "="*55)
print(" TRAINING")
print("="*55)

for epoch in range(1, MAX_EPOCHS + 1):

    # [AR_COMMENT_REMOVED]
    X_vsamp, y_vsamp = make_val_sample(X_val_full, y_val_full,
                                       ratio=VAL_RATIO, seed=epoch)
    val_ds     = NILMDataset(X_vsamp, y_vsamp)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    # ── Train ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b = X_b.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(X_b)
            all_logits.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y_b.numpy())

    probs  = np.concatenate(all_logits)
    labels = np.concatenate(all_labels).astype(int)
    preds  = (probs >= 0.5).astype(int)

    f1  = f1_score(labels, preds, pos_label=1, zero_division=0)
    pr  = precision_score(labels, preds, pos_label=1, zero_division=0)
    rc  = recall_score(labels, preds, pos_label=1, zero_division=0)

    scheduler.step(f1)
    current_lr = optimizer.param_groups[0]['lr']
    history.append({"epoch": epoch, "train_loss": train_loss, "f1": f1})

    flag = ""
    if f1 > best_f1:
        best_f1      = f1
        patience_cnt = 0
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "best_f1":     best_f1,
        }, MODEL_PATH)
        flag = " ← best 💾"
    else:
        patience_cnt += 1

    print(f"  Epoch {epoch:02d}/{MAX_EPOCHS} | "
          f"loss: {train_loss:.4f} | "
          f"F1: {f1:.4f} | P: {pr:.4f} | R: {rc:.4f} | "
          f"lr: {current_lr:.1e}{flag}")

    if patience_cnt >= PATIENCE:
        print(f"\n  ⏹ Early stopping at epoch {epoch} (patience={PATIENCE})")
        break

print(f"\n✅ Training complete | Best F1 (val sample): {best_f1:.4f}")
print(f"   Model saved → {MODEL_PATH}")


# ============================================================
# CELL 6: Full Evaluation + Threshold Sweep
# ============================================================
print("\n" + "="*55)
print(" FULL EVALUATION (623K windows)")
print("="*55)

# [AR_COMMENT_REMOVED]
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"   Loaded best model from epoch {checkpoint['epoch']}")

# ── Full Val Inference ────────────────────────────────────────
full_val_ds     = NILMDataset(X_val_full, y_val_full)
full_val_loader = DataLoader(full_val_ds, batch_size=1024, shuffle=False,
                             num_workers=2, pin_memory=True)

all_probs, all_true = [], []
with torch.no_grad():
    for X_b, y_b in full_val_loader:
        X_b = X_b.to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_true.append(y_b.numpy())

probs_full = np.concatenate(all_probs)
true_full  = np.concatenate(all_true).astype(int)

print(f"\n   Total windows  : {len(probs_full):,}")
print(f"   True ON        : {int(true_full.sum()):,} ({100*true_full.mean():.2f}%)")
print(f"   Prob range     : [{probs_full.min():.4f}, {probs_full.max():.4f}]")

# ── Threshold Sweep ───────────────────────────────────────────
print("\n" + "="*55)
print(" THRESHOLD SWEEP (0.10 → 0.99)")
print("="*55)
print(f"  {'Thresh':>8} | {'F1':>6} | {'Precision':>10} | {'Recall':>8} | {'TP':>7} | {'FP':>7} | {'FN':>7}")
print("  " + "-"*65)

thresholds  = np.arange(0.10, 1.00, 0.05)
best_thresh = 0.5
best_thresh_f1 = 0.0
results = []

for t in thresholds:
    preds = (probs_full >= t).astype(int)
    f1 = f1_score(true_full, preds, pos_label=1, zero_division=0)
    pr = precision_score(true_full, preds, pos_label=1, zero_division=0)
    rc = recall_score(true_full, preds, pos_label=1, zero_division=0)
    cm = confusion_matrix(true_full, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (0, 0, 0, 0)
    results.append({"thresh": t, "f1": f1, "precision": pr, "recall": rc})

    marker = " ←" if f1 > best_thresh_f1 else ""
    if f1 > best_thresh_f1:
        best_thresh_f1 = f1
        best_thresh = t

    print(f"  {t:>8.2f} | {f1:>6.4f} | {pr:>10.4f} | {rc:>8.4f} | "
          f"{tp:>7,} | {fp:>7,} | {fn:>7,}{marker}")

# ── Best Threshold Report ─────────────────────────────────────
print("\n" + "="*55)
print(f" BEST THRESHOLD: {best_thresh:.2f}")
print("="*55)

best_preds = (probs_full >= best_thresh).astype(int)
f1_best = f1_score(true_full, best_preds, pos_label=1, zero_division=0)
pr_best = precision_score(true_full, best_preds, pos_label=1, zero_division=0)
rc_best = recall_score(true_full, best_preds, pos_label=1, zero_division=0)
cm_best = confusion_matrix(true_full, best_preds)

print(f"   F1        : {f1_best:.4f}")
print(f"   Precision : {pr_best:.4f}")
print(f"   Recall    : {rc_best:.4f}")
print(f"\n   Confusion Matrix:")
print(f"   {'':12} Pred OFF  Pred ON")
print(f"   {'True OFF':12} {cm_best[0,0]:>8,}  {cm_best[0,1]:>7,}")
print(f"   {'True ON':12} {cm_best[1,0]:>8,}  {cm_best[1,1]:>7,}")

# ── Comparison Table ──────────────────────────────────────────
print("\n" + "="*55)
print(" COMPARISON WITH OTHER APPLIANCES")
print("="*55)
print(f"  {'Appliance':<18} | {'Architecture':<15} | {'F1':>6} | {'Thresh':>7}")
print("  " + "-"*55)
prev = [
    ("Kettle",          "CNN-BiLSTM",  0.63, 0.99),
    ("Fridge",          "CNN-BiLSTM",  0.53, 0.30),
    ("Microwave",       "DilatedCNN",  0.24, 0.65),
    ("Washing Machine", "CNN-BiLSTM",  0.60, 0.80),
]
for name, arch, f1, thr in prev:
    print(f"  {name:<18} | {arch:<15} | {f1:>6.2f} | {thr:>7.2f}")
print(f"  {'Dishwasher':<18} | {'CNN-BiLSTM':<15} | {f1_best:>6.4f} | {best_thresh:>7.2f}  ← NEW")

print(f"\n✅ Done! Save best_thresh={best_thresh:.2f} for inference.")

Mounted at /content/drive
✅ Drive mounted successfully
✅ Config ready | device: cuda
   Train PKL : /content/drive/MyDrive/new obada/data/preprocessed/dishwasher_train.pkl
   Val PKL   : /content/drive/MyDrive/new obada/data/preprocessed/dishwasher_val.pkl
✅ Dataset helpers defined
✅ CNN-BiLSTM ready | params: 267,521
CNNBiLSTM(
  (cnn): Sequential(
    (0): Conv1d(1, 32, kernel_size=(15,), stride=(1,), padding=(7,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(9,), stride=(1,), padding=(4,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted successfully")


# ============================================================
# CELL 2: Imports & Configuration
# ============================================================
import os, gc, pickle, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────
BASE_PATH        = "/content/drive/MyDrive/new obada/"
PREPROCESSED     = BASE_PATH + "data/preprocessed/"
MODEL_DIR        = BASE_PATH + "models/"
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Appliance ─────────────────────────────────────────────────
APPLIANCE        = "dishwasher"
TRAIN_PKL        = PREPROCESSED + f"{APPLIANCE}_train.pkl"
VAL_PKL          = PREPROCESSED + f"{APPLIANCE}_val.pkl"
MODEL_PATH       = MODEL_DIR    + f"{APPLIANCE}_best.pt"

# ── Hyperparameters ───────────────────────────────────────────
WINDOW_SIZE      = 599
BATCH_SIZE       = 512
LR               = 3e-4
MAX_EPOCHS       = 30
PATIENCE         = 7         # early stopping
VAL_RATIO        = 6         # 1:6  ON:OFF during training validation
RANDOM_SEED      = 42

# ── Device ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config ready | device: {device}")
print(f"   Train PKL : {TRAIN_PKL}")
print(f"   Val PKL   : {VAL_PKL}")


# ============================================================
# CELL 3: Dataset
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y_label):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)   # (N,1,599)
        self.y = torch.tensor(y_label, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def load_pkl(path):
    with open(path, "rb") as f:
        d = pickle.load(f)
    return d["X"], d["y_label"]


def build_val_splits(X_val, y_val, ratio=6, n_splits=10, seed=42):
    """

    Each split: all ON windows (8,757) + different non-overlapping OFF set.

    OFF pool (614,662) divided equally across n_splits:
    """
    rng     = np.random.default_rng(seed)
    on_idx  = np.where(y_val == 1)[0]
    off_idx = np.where(y_val == 0)[0]
    n_off   = min(len(on_idx) * ratio, len(off_idx) // n_splits)

    # [AR_COMMENT_REMOVED]
    shuffled_off = rng.permutation(off_idx)

    splits = []
    for k in range(n_splits):
        start = k * n_off
        end   = start + n_off
        off_k = shuffled_off[start:end]
        idx   = np.sort(np.concatenate([on_idx, off_k]))
        splits.append((X_val[idx], y_val[idx]))

    return splits


print("✅ Dataset helpers defined")


# ============================================================
# CELL 4: Model — CNN-BiLSTM
# ============================================================
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# ─────────────────────────────────────────────────────────────
class CNNBiLSTM(nn.Module):
    def __init__(self, window=599):
        super().__init__()

        # ── CNN Encoder ────────────────────────────────────────
        self.cnn = nn.Sequential(
            # [AR_COMMENT_REMOVED]
            nn.Conv1d(1, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 599 → 299

            # Block 2
            nn.Conv1d(32, 64, kernel_size=9, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 299 → 149

            # Block 3
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 149 → 74
        )

        # ── BiLSTM ─────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )

        # ── Classifier ─────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            # [AR_COMMENT_REMOVED]
        )

    def forward(self, x):
        # x: (B, 1, 599)
        feat = self.cnn(x)                     # (B, 128, 74)
        feat = feat.permute(0, 2, 1)           # (B, 74, 128)
        out, _ = self.lstm(feat)               # (B, 74, 128)
        # [AR_COMMENT_REMOVED]
        out = out[:, -1, :]                    # (B, 128)
        logits = self.classifier(out)          # (B, 1)
        return logits.squeeze(1)               # (B,)


model = CNNBiLSTM(WINDOW_SIZE).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ CNN-BiLSTM ready | params: {total_params:,}")
print(model)


# ============================================================
# CELL 5: Training
# ============================================================
print("\n" + "="*55)
print(" LOADING DATA")
print("="*55)

X_train, y_train = load_pkl(TRAIN_PKL)
X_val_full, y_val_full = load_pkl(VAL_PKL)

print(f"   Train: {X_train.shape} | ON: {int(y_train.sum()):,} ({100*y_train.mean():.1f}%)")
print(f"   Val  : {X_val_full.shape} | ON: {int(y_val_full.sum()):,} ({100*y_val_full.mean():.1f}%)")

# [AR_COMMENT_REMOVED]
N_SPLITS  = 10
val_splits = build_val_splits(X_val_full, y_val_full,
                               ratio=VAL_RATIO, n_splits=N_SPLITS,
                               seed=RANDOM_SEED)
x0, y0 = val_splits[0]
print(f"   Val splits: {N_SPLITS} | each → {len(x0):,} windows | "
      f"ON: {int(y0.sum()):,} | OFF: {int((y0==0).sum()):,}")

train_ds     = NILMDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)

print(f"\n   Train batches: {len(train_loader)}")

# ── Loss & Optimizer ──────────────────────────────────────────
# [AR_COMMENT_REMOVED]
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# ── Training Loop ─────────────────────────────────────────────
best_f1       = 0.0
patience_cnt  = 0
history       = []

print("\n" + "="*55)
print(" TRAINING")
print("="*55)

for epoch in range(1, MAX_EPOCHS + 1):

    # [AR_COMMENT_REMOVED]
    split_idx  = (epoch - 1) % N_SPLITS
    X_vsamp, y_vsamp = val_splits[split_idx]
    val_ds     = NILMDataset(X_vsamp, y_vsamp)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    # ── Train ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b = X_b.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(X_b)
            all_logits.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y_b.numpy())

    probs  = np.concatenate(all_logits)
    labels = np.concatenate(all_labels).astype(int)
    preds  = (probs >= 0.5).astype(int)

    f1  = f1_score(labels, preds, pos_label=1, zero_division=0)
    pr  = precision_score(labels, preds, pos_label=1, zero_division=0)
    rc  = recall_score(labels, preds, pos_label=1, zero_division=0)

    scheduler.step(f1)
    current_lr = optimizer.param_groups[0]['lr']
    history.append({"epoch": epoch, "train_loss": train_loss, "f1": f1})

    flag = ""
    if f1 > best_f1:
        best_f1      = f1
        patience_cnt = 0
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "best_f1":     best_f1,
        }, MODEL_PATH)
        flag = " ← best 💾"
    else:
        patience_cnt += 1

    print(f"  Epoch {epoch:02d}/{MAX_EPOCHS} | "
          f"loss: {train_loss:.4f} | "
          f"F1: {f1:.4f} | P: {pr:.4f} | R: {rc:.4f} | "
          f"lr: {current_lr:.1e}{flag}")

    if patience_cnt >= PATIENCE:
        print(f"\n  ⏹ Early stopping at epoch {epoch} (patience={PATIENCE})")
        break

print(f"\n✅ Training complete | Best F1 (val sample): {best_f1:.4f}")
print(f"   Model saved → {MODEL_PATH}")


# ============================================================
# CELL 6: Full Evaluation + Threshold Sweep
# ============================================================
print("\n" + "="*55)
print(" FULL EVALUATION (623K windows)")
print("="*55)

# [AR_COMMENT_REMOVED]
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"   Loaded best model from epoch {checkpoint['epoch']}")

# ── Full Val Inference ────────────────────────────────────────
full_val_ds     = NILMDataset(X_val_full, y_val_full)
full_val_loader = DataLoader(full_val_ds, batch_size=1024, shuffle=False,
                             num_workers=2, pin_memory=True)

all_probs, all_true = [], []
with torch.no_grad():
    for X_b, y_b in full_val_loader:
        X_b = X_b.to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_true.append(y_b.numpy())

probs_full = np.concatenate(all_probs)
true_full  = np.concatenate(all_true).astype(int)

print(f"\n   Total windows  : {len(probs_full):,}")
print(f"   True ON        : {int(true_full.sum()):,} ({100*true_full.mean():.2f}%)")
print(f"   Prob range     : [{probs_full.min():.4f}, {probs_full.max():.4f}]")

# ── Threshold Sweep ───────────────────────────────────────────
print("\n" + "="*55)
print(" THRESHOLD SWEEP (0.10 → 0.99)")
print("="*55)
print(f"  {'Thresh':>8} | {'F1':>6} | {'Precision':>10} | {'Recall':>8} | {'TP':>7} | {'FP':>7} | {'FN':>7}")
print("  " + "-"*65)

thresholds  = np.arange(0.10, 1.00, 0.05)
best_thresh = 0.5
best_thresh_f1 = 0.0
results = []

for t in thresholds:
    preds = (probs_full >= t).astype(int)
    f1 = f1_score(true_full, preds, pos_label=1, zero_division=0)
    pr = precision_score(true_full, preds, pos_label=1, zero_division=0)
    rc = recall_score(true_full, preds, pos_label=1, zero_division=0)
    cm = confusion_matrix(true_full, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (0, 0, 0, 0)
    results.append({"thresh": t, "f1": f1, "precision": pr, "recall": rc})

    marker = " ←" if f1 > best_thresh_f1 else ""
    if f1 > best_thresh_f1:
        best_thresh_f1 = f1
        best_thresh = t

    print(f"  {t:>8.2f} | {f1:>6.4f} | {pr:>10.4f} | {rc:>8.4f} | "
          f"{tp:>7,} | {fp:>7,} | {fn:>7,}{marker}")

# ── Best Threshold Report ─────────────────────────────────────
print("\n" + "="*55)
print(f" BEST THRESHOLD: {best_thresh:.2f}")
print("="*55)

best_preds = (probs_full >= best_thresh).astype(int)
f1_best = f1_score(true_full, best_preds, pos_label=1, zero_division=0)
pr_best = precision_score(true_full, best_preds, pos_label=1, zero_division=0)
rc_best = recall_score(true_full, best_preds, pos_label=1, zero_division=0)
cm_best = confusion_matrix(true_full, best_preds)

print(f"   F1        : {f1_best:.4f}")
print(f"   Precision : {pr_best:.4f}")
print(f"   Recall    : {rc_best:.4f}")
print(f"\n   Confusion Matrix:")
print(f"   {'':12} Pred OFF  Pred ON")
print(f"   {'True OFF':12} {cm_best[0,0]:>8,}  {cm_best[0,1]:>7,}")
print(f"   {'True ON':12} {cm_best[1,0]:>8,}  {cm_best[1,1]:>7,}")

# ── Comparison Table ──────────────────────────────────────────
print("\n" + "="*55)
print(" COMPARISON WITH OTHER APPLIANCES")
print("="*55)
print(f"  {'Appliance':<18} | {'Architecture':<15} | {'F1':>6} | {'Thresh':>7}")
print("  " + "-"*55)
prev = [
    ("Kettle",          "CNN-BiLSTM",  0.63, 0.99),
    ("Fridge",          "CNN-BiLSTM",  0.53, 0.30),
    ("Microwave",       "DilatedCNN",  0.24, 0.65),
    ("Washing Machine", "CNN-BiLSTM",  0.60, 0.80),
]
for name, arch, f1, thr in prev:
    print(f"  {name:<18} | {arch:<15} | {f1:>6.2f} | {thr:>7.2f}")
print(f"  {'Dishwasher':<18} | {'CNN-BiLSTM':<15} | {f1_best:>6.4f} | {best_thresh:>7.2f}  ← NEW")

print(f"\n✅ Done! Save best_thresh={best_thresh:.2f} for inference.")

Mounted at /content/drive
✅ Drive mounted successfully
✅ Config ready | device: cuda
   Train PKL : /content/drive/MyDrive/new obada/data/preprocessed/dishwasher_train.pkl
   Val PKL   : /content/drive/MyDrive/new obada/data/preprocessed/dishwasher_val.pkl
✅ Dataset helpers defined
✅ CNN-BiLSTM ready | params: 267,521
CNNBiLSTM(
  (cnn): Sequential(
    (0): Conv1d(1, 32, kernel_size=(15,), stride=(1,), padding=(7,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(9,), stride=(1,), padding=(4,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted successfully")


# ============================================================
# CELL 2: Imports & Configuration
# ============================================================
import os, gc, pickle, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────
BASE_PATH        = "/content/drive/MyDrive/new obada/"
PREPROCESSED     = BASE_PATH + "data/preprocessed/"
MODEL_DIR        = BASE_PATH + "models/"
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Appliance ─────────────────────────────────────────────────
APPLIANCE        = "dishwasher"
TRAIN_PKL        = PREPROCESSED + f"{APPLIANCE}_train.pkl"
VAL_PKL          = PREPROCESSED + f"{APPLIANCE}_val.pkl"
MODEL_PATH       = MODEL_DIR    + f"{APPLIANCE}_best.pt"

# ── Hyperparameters ───────────────────────────────────────────
WINDOW_SIZE      = 599
BATCH_SIZE       = 512
LR               = 1e-4
MAX_EPOCHS       = 40
PATIENCE         = 10
VAL_RATIO        = 6              # 1:6  ON:OFF
N_SPLITS         = 10             # K fixed val splits
RANDOM_SEED      = 42

# ── Focal Loss γ ─────────────────────────────────────────────
# [AR_COMMENT_REMOVED]
FOCAL_GAMMA      = 2.0
FOCAL_ALPHA      = 0.75

# ── Device ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config ready | device: {device}")
print(f"   Loss: Focal (γ={FOCAL_GAMMA}, α={FOCAL_ALPHA})")
print(f"   LR: {LR} | Epochs: {MAX_EPOCHS} | Patience: {PATIENCE}")


# ============================================================
# CELL 3: Dataset & Val Splits
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y_label):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y_label, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def load_pkl(path):
    with open(path, "rb") as f:
        d = pickle.load(f)
    return d["X"], d["y_label"]


def build_val_splits(X_val, y_val, ratio=6, n_splits=10, seed=42):
    """
    """
    rng          = np.random.default_rng(seed)
    on_idx       = np.where(y_val == 1)[0]
    off_idx      = np.where(y_val == 0)[0]
    n_off        = min(len(on_idx) * ratio, len(off_idx) // n_splits)
    shuffled_off = rng.permutation(off_idx)

    splits = []
    for k in range(n_splits):
        off_k = shuffled_off[k * n_off : (k+1) * n_off]
        idx   = np.sort(np.concatenate([on_idx, off_k]))
        splits.append((X_val[idx], y_val[idx]))
    return splits


print("✅ Dataset helpers defined")


# ============================================================
# CELL 4: Focal Loss + Model
# ============================================================
class FocalLoss(nn.Module):
    """

    FL(p) = -alpha * (1-p)^gamma * log(p)    for positive class (ON)
    FL(p) = -(1-alpha) * p^gamma * log(1-p)  for negative class (OFF)

    gamma: higher value = more focus on hard (misclassified) examples
               gamma=0 -> standard BCE,  gamma=2 -> standard Focal Loss
    alpha: weight of the positive class — compensates class imbalance
    """
    def __init__(self, gamma=2.0, alpha=0.75, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        bce   = nn.functional.binary_cross_entropy_with_logits(
                    logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        # [AR_COMMENT_REMOVED]
        p_t   = probs * targets + (1 - probs) * (1 - targets)
        # [AR_COMMENT_REMOVED]
        a_t   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss  = a_t * (1 - p_t) ** self.gamma * bce

        if self.reduction == 'mean':
            return loss.mean()
        return loss.sum()


class CNNBiLSTM(nn.Module):
    """
    """
    def __init__(self, window=599):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 599 → 299

            nn.Conv1d(32, 64, kernel_size=9, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 299 → 149

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),                    # 149 → 74
        )

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.4,
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        feat = self.cnn(x)
        feat = feat.permute(0, 2, 1)
        out, _ = self.lstm(feat)
        out = out[:, -1, :]
        return self.classifier(out).squeeze(1)


model = CNNBiLSTM(WINDOW_SIZE).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ CNN-BiLSTM ready | params: {total_params:,}")


# ============================================================
# CELL 5: Training
# ============================================================
print("\n" + "="*55)
print(" LOADING DATA")
print("="*55)

X_train, y_train = load_pkl(TRAIN_PKL)
X_val_full, y_val_full = load_pkl(VAL_PKL)

print(f"   Train: {X_train.shape} | ON: {int(y_train.sum()):,} ({100*y_train.mean():.1f}%)")
print(f"   Val  : {X_val_full.shape} | ON: {int(y_val_full.sum()):,} ({100*y_val_full.mean():.1f}%)")

val_splits = build_val_splits(X_val_full, y_val_full,
                               ratio=VAL_RATIO, n_splits=N_SPLITS,
                               seed=RANDOM_SEED)
x0, y0 = val_splits[0]
print(f"   Val splits: {N_SPLITS} | each → {len(x0):,} windows | "
      f"ON: {int(y0.sum()):,} | OFF: {int((y0==0).sum()):,}")

train_ds     = NILMDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)

print(f"\n   Train batches: {len(train_loader)}")

# ── Loss, Optimizer, Scheduler ───────────────────────────────
criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)

# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
)

scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# ── Training Loop ─────────────────────────────────────────────
best_f1      = 0.0
patience_cnt = 0
history      = []

print("\n" + "="*55)
print(" TRAINING")
print("="*55)

for epoch in range(1, MAX_EPOCHS + 1):

    # [AR_COMMENT_REMOVED]
    split_idx        = (epoch - 1) % N_SPLITS
    X_vsamp, y_vsamp = val_splits[split_idx]
    val_ds           = NILMDataset(X_vsamp, y_vsamp)
    val_loader       = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=2, pin_memory=True)

    # ── Train ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b = X_b.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(X_b)
            all_logits.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y_b.numpy())

    probs  = np.concatenate(all_logits)
    labels = np.concatenate(all_labels).astype(int)
    preds  = (probs >= 0.5).astype(int)

    f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
    pr = precision_score(labels, preds, pos_label=1, zero_division=0)
    rc = recall_score(labels, preds, pos_label=1, zero_division=0)

    history.append({"epoch": epoch, "train_loss": train_loss, "f1": f1})

    flag = ""
    if f1 > best_f1:
        best_f1      = f1
        patience_cnt = 0
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "best_f1":     best_f1,
        }, MODEL_PATH)
        flag = " ← best 💾"
    else:
        patience_cnt += 1

    print(f"  Epoch {epoch:02d}/{MAX_EPOCHS} | "
          f"loss: {train_loss:.4f} | "
          f"F1: {f1:.4f} | P: {pr:.4f} | R: {rc:.4f} | "
          f"lr: {current_lr:.1e}{flag}")

    if patience_cnt >= PATIENCE:
        print(f"\n  ⏹ Early stopping at epoch {epoch} (patience={PATIENCE})")
        break

print(f"\n✅ Training complete | Best F1 (val sample): {best_f1:.4f}")
print(f"   Model saved → {MODEL_PATH}")


# ============================================================
# CELL 6: Full Evaluation + Threshold Sweep
# ============================================================
print("\n" + "="*55)
print(" FULL EVALUATION (623K windows)")
print("="*55)

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"   Loaded best model from epoch {checkpoint['epoch']}")

full_val_ds     = NILMDataset(X_val_full, y_val_full)
full_val_loader = DataLoader(full_val_ds, batch_size=1024, shuffle=False,
                             num_workers=2, pin_memory=True)

all_probs, all_true = [], []
with torch.no_grad():
    for X_b, y_b in full_val_loader:
        X_b = X_b.to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_true.append(y_b.numpy())

probs_full = np.concatenate(all_probs)
true_full  = np.concatenate(all_true).astype(int)

print(f"\n   Total windows  : {len(probs_full):,}")
print(f"   True ON        : {int(true_full.sum()):,} ({100*true_full.mean():.2f}%)")
print(f"   Prob range     : [{probs_full.min():.4f}, {probs_full.max():.4f}]")

# ── Threshold Sweep ───────────────────────────────────────────
print("\n" + "="*55)
print(" THRESHOLD SWEEP (0.10 → 0.99)")
print("="*55)
print(f"  {'Thresh':>8} | {'F1':>6} | {'Precision':>10} | {'Recall':>8} | {'TP':>7} | {'FP':>7} | {'FN':>7}")
print("  " + "-"*65)

thresholds     = np.arange(0.10, 1.00, 0.05)
best_thresh    = 0.5
best_thresh_f1 = 0.0

for t in thresholds:
    preds = (probs_full >= t).astype(int)
    f1    = f1_score(true_full, preds, pos_label=1, zero_division=0)
    pr    = precision_score(true_full, preds, pos_label=1, zero_division=0)
    rc    = recall_score(true_full, preds, pos_label=1, zero_division=0)
    cm    = confusion_matrix(true_full, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    marker = " ←" if f1 > best_thresh_f1 else ""
    if f1 > best_thresh_f1:
        best_thresh_f1 = f1
        best_thresh    = t

    print(f"  {t:>8.2f} | {f1:>6.4f} | {pr:>10.4f} | {rc:>8.4f} | "
          f"{tp:>7,} | {fp:>7,} | {fn:>7,}{marker}")

# ── Best Threshold Report ─────────────────────────────────────
print("\n" + "="*55)
print(f" BEST THRESHOLD: {best_thresh:.2f}")
print("="*55)

best_preds = (probs_full >= best_thresh).astype(int)
f1_best    = f1_score(true_full, best_preds, pos_label=1, zero_division=0)
pr_best    = precision_score(true_full, best_preds, pos_label=1, zero_division=0)
rc_best    = recall_score(true_full, best_preds, pos_label=1, zero_division=0)
cm_best    = confusion_matrix(true_full, best_preds)

print(f"   F1        : {f1_best:.4f}")
print(f"   Precision : {pr_best:.4f}")
print(f"   Recall    : {rc_best:.4f}")
print(f"\n   Confusion Matrix:")
print(f"   {'':12} Pred OFF  Pred ON")
print(f"   {'True OFF':12} {cm_best[0,0]:>8,}  {cm_best[0,1]:>7,}")
print(f"   {'True ON':12} {cm_best[1,0]:>8,}  {cm_best[1,1]:>7,}")

# ── Comparison Table ──────────────────────────────────────────
print("\n" + "="*55)
print(" COMPARISON WITH OTHER APPLIANCES")
print("="*55)
print(f"  {'Appliance':<18} | {'Architecture':<15} | {'F1':>6} | {'Thresh':>7}")
print("  " + "-"*55)
for name, arch, f1, thr in [
    ("Kettle",          "CNN-BiLSTM", 0.63, 0.99),
    ("Fridge",          "CNN-BiLSTM", 0.53, 0.30),
    ("Microwave",       "DilatedCNN", 0.24, 0.65),
    ("Washing Machine", "CNN-BiLSTM", 0.60, 0.80),
]:
    print(f"  {name:<18} | {arch:<15} | {f1:>6.2f} | {thr:>7.2f}")
print(f"  {'Dishwasher':<18} | {'CNN-BiLSTM':<15} | {f1_best:>6.4f} | {best_thresh:>7.2f}  ← NEW")

print(f"\n✅ Done! Save best_thresh={best_thresh:.2f} for inference.")

Mounted at /content/drive
✅ Drive mounted successfully
✅ Config ready | device: cuda
   Loss: Focal (γ=2.0, α=0.75)
   LR: 0.0001 | Epochs: 40 | Patience: 10
✅ Dataset helpers defined
✅ CNN-BiLSTM ready | params: 267,521

 LOADING DATA
   Train: (215858, 599) | ON: 107,929 (50.0%)
   Val  : (623419, 599) | ON: 8,757 (1.4%)
   Val splits: 10 | each → 61,299 windows | ON: 8,757 | OFF: 52,542

   Train batches: 422

 TRAINING
  Epoch 01/40 | loss: 0.0566 | F1: 0.7419 | P: 0.6665 | R: 0.8366 | lr: 1.0e-04 ← best 💾
  Epoch 02/40 | loss: 0.0219 | F1: 0.7277 | P: 0.6630 | R: 0.8063 | lr: 9.9e-05
  Epoch 03/40 | loss: 0.0171 | F1: 0.7217 | P: 0.6546 | R: 0.8042 | lr: 9.9e-05
  Epoch 04/40 | loss: 0.0145 | F1: 0.7503 | P: 0.6922 | R: 0.8189 | lr: 9.8e-05 ← best 💾
  Epoch 05/40 | loss: 0.0126 | F1: 0.7783 | P: 0.7228 | R: 0.8430 | lr: 9.6e-05 ← best 💾
  Epoch 06/40 | loss: 0.0115 | F1: 0.5853 | P: 0.6459 | R: 0.5350 | lr: 9.5e-05
  Epoch 07/40 | loss: 0.0108 | F1: 0.7782 | P: 0.6715 | R: 0.9252 

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted successfully")


# ============================================================
# CELL 2: Imports & Configuration
# ============================================================
import os, gc, pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

BASE_PATH    = "/content/drive/MyDrive/new obada/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODEL_DIR    = BASE_PATH + "models/"
os.makedirs(MODEL_DIR, exist_ok=True)

APPLIANCE    = "dishwasher"
TRAIN_PKL    = PREPROCESSED + f"{APPLIANCE}_train.pkl"
VAL_PKL      = PREPROCESSED + f"{APPLIANCE}_val.pkl"
MODEL_PATH   = MODEL_DIR    + f"{APPLIANCE}_transformer_best.pt"

WINDOW_SIZE  = 599
BATCH_SIZE   = 256
LR           = 5e-4
MAX_EPOCHS   = 40
PATIENCE     = 10
VAL_RATIO    = 6
N_SPLITS     = 10
RANDOM_SEED  = 42

# Focal Loss
FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.75

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config ready | device: {device}")
print(f"   Architecture: Transformer Seq2Point")
print(f"   Loss: Focal (γ={FOCAL_GAMMA}, α={FOCAL_ALPHA})")


# ============================================================
# CELL 3: Dataset & Val Splits
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y_label):
        # (N, 599) → (N, 599, 1): Transformer expects (batch, seq, features)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(2)
        self.y = torch.tensor(y_label, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def load_pkl(path):
    with open(path, "rb") as f:
        d = pickle.load(f)
    return d["X"], d["y_label"]


def build_val_splits(X_val, y_val, ratio=6, n_splits=10, seed=42):
    rng          = np.random.default_rng(seed)
    on_idx       = np.where(y_val == 1)[0]
    off_idx      = np.where(y_val == 0)[0]
    n_off        = min(len(on_idx) * ratio, len(off_idx) // n_splits)
    shuffled_off = rng.permutation(off_idx)
    splits = []
    for k in range(n_splits):
        off_k = shuffled_off[k * n_off : (k+1) * n_off]
        idx   = np.sort(np.concatenate([on_idx, off_k]))
        splits.append((X_val[idx], y_val[idx]))
    return splits


print("✅ Dataset helpers defined")


# ============================================================
# CELL 4: Model — Transformer Seq2Point
# ============================================================
# [AR_COMMENT_REMOVED]
# ─────────────────────────────────────────────────────────
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# [AR_COMMENT_REMOVED]
# ─────────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    """
    """
    def __init__(self, d_model, max_len=600, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, seq_len, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])


class NILMTransformer(nn.Module):
    """

    Architecture:
    1. Input projection: 1 -> d_model (each power reading -> vector)
    5. Classifier: linear → logit

    CLS token (inspired by BERT):
    """
    def __init__(self,
                 window=599,
                 d_model=64,
                 nhead=4,
                 num_layers=3,
                 dim_feedforward=256,
                 dropout=0.1):
        super().__init__()

        # ── Input projection ──────────────────────────────────
        # [AR_COMMENT_REMOVED]
        self.input_proj = nn.Sequential(
            nn.Linear(1, d_model),
            nn.LayerNorm(d_model),
        )

        # ── CLS token ─────────────────────────────────────────
        # [AR_COMMENT_REMOVED]
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # ── Positional Encoding ───────────────────────────────
        self.pos_enc = PositionalEncoding(d_model, max_len=window+1, dropout=dropout)

        # ── Transformer Encoder ───────────────────────────────
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,       # (batch, seq, features)
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        # ── Classifier ────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
            # [AR_COMMENT_REMOVED]
        )

    def forward(self, x):
        # x: (B, 599, 1)
        B = x.size(0)

        # Project input
        x = self.input_proj(x)                       # (B, 599, d_model)

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)       # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)             # (B, 600, d_model)

        # Positional encoding
        x = self.pos_enc(x)                          # (B, 600, d_model)

        # Transformer
        x = self.transformer(x)                      # (B, 600, d_model)

        # [AR_COMMENT_REMOVED]
        cls_out = x[:, 0, :]                         # (B, d_model)

        # Classify
        return self.classifier(cls_out).squeeze(1)   # (B,)


model = NILMTransformer(
    window=WINDOW_SIZE,
    d_model=64,
    nhead=4,
    num_layers=3,
    dim_feedforward=256,
    dropout=0.1,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ NILMTransformer ready | params: {total_params:,}")
print(model)


# ============================================================
# CELL 5: Focal Loss
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce   = nn.functional.binary_cross_entropy_with_logits(
                    logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t   = probs * targets + (1 - probs) * (1 - targets)
        a_t   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss  = a_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()


print("✅ FocalLoss defined")


# ============================================================
# CELL 6: Training
# ============================================================
print("\n" + "="*55)
print(" LOADING DATA")
print("="*55)

X_train, y_train   = load_pkl(TRAIN_PKL)
X_val_full, y_val_full = load_pkl(VAL_PKL)

print(f"   Train: {X_train.shape} | ON: {int(y_train.sum()):,} ({100*y_train.mean():.1f}%)")
print(f"   Val  : {X_val_full.shape} | ON: {int(y_val_full.sum()):,} ({100*y_val_full.mean():.1f}%)")

val_splits = build_val_splits(X_val_full, y_val_full,
                               ratio=VAL_RATIO, n_splits=N_SPLITS,
                               seed=RANDOM_SEED)
x0, y0 = val_splits[0]
print(f"   Val splits: {N_SPLITS} | each → {len(x0):,} windows | "
      f"ON: {int(y0.sum()):,} | OFF: {int((y0==0).sum()):,}")

train_ds     = NILMDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
print(f"\n   Train batches: {len(train_loader)}")

# ── Loss, Optimizer, Scheduler ───────────────────────────────
criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)

# Warmup + Cosine Decay
# [AR_COMMENT_REMOVED]
def lr_lambda(epoch):
    warmup_epochs = 3
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, MAX_EPOCHS - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# ── Training Loop ─────────────────────────────────────────────
best_f1      = 0.0
patience_cnt = 0

print("\n" + "="*55)
print(" TRAINING")
print("="*55)

for epoch in range(1, MAX_EPOCHS + 1):

    split_idx        = (epoch - 1) % N_SPLITS
    X_vsamp, y_vsamp = val_splits[split_idx]
    val_ds           = NILMDataset(X_vsamp, y_vsamp)
    val_loader       = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=2, pin_memory=True)

    # ── Train ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b = X_b.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(X_b)
            all_logits.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y_b.numpy())

    probs  = np.concatenate(all_logits)
    labels = np.concatenate(all_labels).astype(int)
    preds  = (probs >= 0.5).astype(int)

    f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
    pr = precision_score(labels, preds, pos_label=1, zero_division=0)
    rc = recall_score(labels, preds, pos_label=1, zero_division=0)

    flag = ""
    if f1 > best_f1:
        best_f1      = f1
        patience_cnt = 0
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "best_f1":     best_f1,
        }, MODEL_PATH)
        flag = " ← best 💾"
    else:
        patience_cnt += 1

    print(f"  Epoch {epoch:02d}/{MAX_EPOCHS} | "
          f"loss: {train_loss:.4f} | "
          f"F1: {f1:.4f} | P: {pr:.4f} | R: {rc:.4f} | "
          f"lr: {current_lr:.1e}{flag}")

    if patience_cnt >= PATIENCE:
        print(f"\n  ⏹ Early stopping at epoch {epoch} (patience={PATIENCE})")
        break

print(f"\n✅ Training complete | Best F1 (val sample): {best_f1:.4f}")
print(f"   Model saved → {MODEL_PATH}")


# ============================================================
# CELL 7: Full Evaluation + Threshold Sweep
# ============================================================
print("\n" + "="*55)
print(" FULL EVALUATION (623K windows)")
print("="*55)

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"   Loaded best model from epoch {checkpoint['epoch']}")

full_val_ds     = NILMDataset(X_val_full, y_val_full)
full_val_loader = DataLoader(full_val_ds, batch_size=512, shuffle=False,
                             num_workers=2, pin_memory=True)

all_probs, all_true = [], []
with torch.no_grad():
    for X_b, y_b in full_val_loader:
        X_b = X_b.to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_true.append(y_b.numpy())

probs_full = np.concatenate(all_probs)
true_full  = np.concatenate(all_true).astype(int)

print(f"\n   Total windows : {len(probs_full):,}")
print(f"   True ON       : {int(true_full.sum()):,} ({100*true_full.mean():.2f}%)")
print(f"   Prob range    : [{probs_full.min():.4f}, {probs_full.max():.4f}]")

print("\n" + "="*55)
print(" THRESHOLD SWEEP (0.10 → 0.99)")
print("="*55)
print(f"  {'Thresh':>8} | {'F1':>6} | {'Precision':>10} | {'Recall':>8} | {'TP':>7} | {'FP':>7} | {'FN':>7}")
print("  " + "-"*65)

best_thresh    = 0.5
best_thresh_f1 = 0.0

for t in np.arange(0.10, 1.00, 0.05):
    preds = (probs_full >= t).astype(int)
    f1    = f1_score(true_full, preds, pos_label=1, zero_division=0)
    pr    = precision_score(true_full, preds, pos_label=1, zero_division=0)
    rc    = recall_score(true_full, preds, pos_label=1, zero_division=0)
    cm    = confusion_matrix(true_full, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    marker = " ←" if f1 > best_thresh_f1 else ""
    if f1 > best_thresh_f1:
        best_thresh_f1 = f1
        best_thresh    = t

    print(f"  {t:>8.2f} | {f1:>6.4f} | {pr:>10.4f} | {rc:>8.4f} | "
          f"{tp:>7,} | {fp:>7,} | {fn:>7,}{marker}")

print("\n" + "="*55)
print(f" BEST THRESHOLD: {best_thresh:.2f}")
print("="*55)

best_preds = (probs_full >= best_thresh).astype(int)
f1_best    = f1_score(true_full, best_preds, pos_label=1, zero_division=0)
pr_best    = precision_score(true_full, best_preds, pos_label=1, zero_division=0)
rc_best    = recall_score(true_full, best_preds, pos_label=1, zero_division=0)
cm_best    = confusion_matrix(true_full, best_preds)

print(f"   F1        : {f1_best:.4f}")
print(f"   Precision : {pr_best:.4f}")
print(f"   Recall    : {rc_best:.4f}")
print(f"\n   Confusion Matrix:")
print(f"   {'':12} Pred OFF  Pred ON")
print(f"   {'True OFF':12} {cm_best[0,0]:>8,}  {cm_best[0,1]:>7,}")
print(f"   {'True ON':12} {cm_best[1,0]:>8,}  {cm_best[1,1]:>7,}")

print("\n" + "="*55)
print(" COMPARISON")
print("="*55)
baseline = 0.574
arrow    = "↑" if f1_best > baseline else "↓"
diff     = abs(f1_best - baseline)
print(f"   CNN-BiLSTM (baseline) : F1 = {baseline:.4f}")
print(f"   Transformer (this)    : F1 = {f1_best:.4f}  {arrow} {diff:.4f}")
print(f"\n✅ Done! Save best_thresh={best_thresh:.2f} for inference.")

Mounted at /content/drive
✅ Drive mounted successfully
✅ Config ready | device: cuda
   Architecture: Transformer Seq2Point
   Loss: Focal (γ=2.0, α=0.75)
✅ Dataset helpers defined
✅ NILMTransformer ready | params: 152,513
NILMTransformer(
  (input_proj): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (pos_enc): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (n

In [ ]:
# ============================================================
# CELL 1: Mount & Install
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess
subprocess.run(["pip", "install", "pyts", "--quiet"])
print("✅ Ready")


# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, gc, pickle, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from scipy.signal import spectrogram as scipy_spectrogram
warnings.filterwarnings("ignore")

BASE_PATH    = "/content/drive/MyDrive/new obada/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODEL_DIR    = BASE_PATH + "models/"
os.makedirs(MODEL_DIR, exist_ok=True)

APPLIANCE   = "dishwasher"
TRAIN_PKL   = PREPROCESSED + f"{APPLIANCE}_train.pkl"
VAL_PKL     = PREPROCESSED + f"{APPLIANCE}_val.pkl"

WINDOW_SIZE  = 599
BATCH_SIZE   = 128
IMG_SIZE     = 64       # 64×64 pixels
LR           = 1e-4
MAX_EPOCHS   = 20
PATIENCE     = 5
FULL_EVAL_EVERY = 2
RANDOM_SEED  = 42
FOCAL_GAMMA  = 2.97
FOCAL_ALPHA  = 0.578

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config | device: {device}")
print(f"   Image size: {IMG_SIZE}×{IMG_SIZE}")
print(f"   Conversions: Spectrogram | GAF | Recurrence Plot")


# ============================================================
# CELL 3: Signal → Image Converters
# ============================================================
def to_spectrogram(signal, img_size=64):
    """
    STFT Spectrogram:
    signal (599,) → freq × time matrix → resize → (img_size, img_size)
    """
    nperseg = min(32, len(signal) // 4)
    f, t, Sxx = scipy_spectrogram(signal, nperseg=nperseg,
                                   noverlap=nperseg//2)
    Sxx = np.log1p(Sxx)
    # Normalize
    if Sxx.max() > Sxx.min():
        Sxx = (Sxx - Sxx.min()) / (Sxx.max() - Sxx.min())
    # Resize to img_size × img_size
    from PIL import Image
    img = Image.fromarray((Sxx * 255).astype(np.uint8))
    img = img.resize((img_size, img_size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0


def to_gaf(signal, img_size=64):
    """
    Gramian Angular Field (GAF):
    G[i,j] = cos(φ_i + φ_j)
    """
    # [AR_COMMENT_REMOVED]
    n = img_size
    idx = np.linspace(0, len(signal)-1, n).astype(int)
    s   = signal[idx]

    # [AR_COMMENT_REMOVED]
    s_min, s_max = s.min(), s.max()
    if s_max > s_min:
        s = 2 * (s - s_min) / (s_max - s_min) - 1
    else:
        s = np.zeros_like(s)
    s = np.clip(s, -1, 1)

    # Arccos → angular representation
    phi = np.arccos(s)

    # Gramian: cos(φ_i + φ_j)
    gaf = np.cos(phi[:, None] + phi[None, :])

    # [AR_COMMENT_REMOVED]
    gaf = (gaf + 1) / 2
    return gaf.astype(np.float32)


def to_recurrence(signal, img_size=64, threshold=0.2):
    """
    Recurrence Plot:
    R[i,j] = 1 if |s_i - s_j| < threshold
    """
    n   = img_size
    idx = np.linspace(0, len(signal)-1, n).astype(int)
    s   = signal[idx]

    # Normalize
    s_min, s_max = s.min(), s.max()
    if s_max > s_min:
        s = (s - s_min) / (s_max - s_min)

    # Distance matrix
    diff = np.abs(s[:, None] - s[None, :])
    rp   = (diff < threshold).astype(np.float32)
    return rp


print("✅ Converters defined")


# ============================================================
# CELL 4: Dataset (on-the-fly conversion)
# ============================================================
class NILMImageDataset(Dataset):
    """
    Converts each window to a 3-channel (RGB) image on-the-fly
    """
    def __init__(self, X, y_label, mode="spectrogram",
                 img_size=64, transform=None):
        self.X         = X
        self.y         = torch.tensor(y_label, dtype=torch.float32)
        self.mode      = mode
        self.img_size  = img_size
        self.transform = transform

        self.converters = {
            "spectrogram": to_spectrogram,
            "gaf":         to_gaf,
            "recurrence":  to_recurrence,
        }

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        signal = self.X[i]  # (599,)
        img    = self.converters[self.mode](signal, self.img_size)
        # [AR_COMMENT_REMOVED]
        img_t  = torch.tensor(img).unsqueeze(0).repeat(3, 1, 1)
        if self.transform:
            img_t = self.transform(img_t)
        return img_t, self.y[i]


def load_pkl(path):
    with open(path, "rb") as f:
        d = pickle.load(f)
    return d["X"], d["y_label"]


print("✅ Dataset defined")


# ============================================================
# CELL 5: ResNet18 Fine-tuning Setup
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.97, alpha=0.578):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce   = nn.functional.binary_cross_entropy_with_logits(
                    logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t   = probs * targets + (1 - probs) * (1 - targets)
        a_t   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()


def build_resnet(freeze_backbone=False):
    """
    freeze_backbone=True:  freeze backbone, train head only (faster)
    freeze_backbone=False: fine-tune entire model (better results)
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # [AR_COMMENT_REMOVED]
    in_features = model.fc.in_features  # 512
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 1),
        # [AR_COMMENT_REMOVED]
    )
    return model


def full_eval(model, X_val, y_val, mode, batch_size=256):
    ds     = NILMImageDataset(X_val, np.zeros(len(X_val)), mode=mode,
                               img_size=IMG_SIZE)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)
    probs  = []
    model.eval()
    with torch.no_grad():
        for X_b, _ in loader:
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(X_b.to(device)).squeeze(1)
            probs.append(torch.sigmoid(logits).cpu().numpy())
    probs  = np.concatenate(probs)
    labels = y_val.astype(int)
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.10, 0.99, 0.05):
        f1 = f1_score(labels, (probs >= t).astype(int),
                      pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return probs, best_f1, best_t


print("✅ Model & helpers defined")


# ============================================================
# CELL 6: Train One Mode
# ============================================================
def train_mode(mode, X_train, y_train, X_val_full, y_val_full):
    print(f"\n{'='*55}")
    print(f" MODE: {mode.upper()}")
    print(f"{'='*55}")

    model_path = MODEL_DIR + f"{APPLIANCE}_resnet_{mode}.pt"

    train_ds     = NILMImageDataset(X_train, y_train, mode=mode,
                                     img_size=IMG_SIZE)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                               shuffle=True, num_workers=4,
                               pin_memory=True)

    model     = build_resnet(freeze_backbone=False).to(device)
    criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)
    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=LR, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    best_f1      = 0.0
    best_thresh  = 0.5
    patience_cnt = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                loss = criterion(model(X_b).squeeze(1), y_b)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        scheduler.step()
        lr_now = optimizer.param_groups[0]['lr']

        if epoch % FULL_EVAL_EVERY == 0 or epoch == 1:
            _, f1, best_t = full_eval(model, X_val_full,
                                       y_val_full, mode)
            flag = ""
            if f1 > best_f1:
                best_f1      = f1
                best_thresh  = best_t
                patience_cnt = 0
                torch.save({
                    "epoch":       epoch,
                    "model_state": model.state_dict(),
                    "best_f1":     best_f1,
                    "best_thresh": best_thresh,
                    "mode":        mode,
                }, model_path)
                flag = " ← best 💾"
            else:
                patience_cnt += 1

            print(f"  Epoch {epoch:02d}/{MAX_EPOCHS} | "
                  f"loss: {train_loss:.4f} | "
                  f"✦ F1: {f1:.4f} (t={best_t:.2f}) | "
                  f"lr: {lr_now:.1e}{flag}")
        else:
            print(f"  Epoch {epoch:02d}/{MAX_EPOCHS} | "
                  f"loss: {train_loss:.4f} | lr: {lr_now:.1e}")

        if patience_cnt >= PATIENCE:
            print(f"  ⏹ Early stopping at epoch {epoch}")
            break

    del model, train_loader, train_ds
    gc.collect()
    torch.cuda.empty_cache()

    print(f"  Best F1 ({mode}): {best_f1:.4f} @ thresh={best_thresh:.2f}")
    return best_f1, best_thresh, model_path


# ============================================================
# CELL 7: Run All 3 Modes & Compare
# ============================================================
print("\nLoading data...")
X_train, y_train       = load_pkl(TRAIN_PKL)
X_val_full, y_val_full = load_pkl(VAL_PKL)
print(f"   Train: {X_train.shape} | Val: {X_val_full.shape}")

results = {}
for mode in ["spectrogram", "gaf", "recurrence"]:
    f1, thresh, path = train_mode(mode, X_train, y_train,
                                   X_val_full, y_val_full)
    results[mode] = {"f1": f1, "thresh": thresh, "path": path}

print("\n" + "="*55)
print(" FINAL COMPARISON — ResNet18 Approaches")
print("="*55)
print(f"  {'Mode':<15} | {'F1':>6} | {'Thresh':>7}")
print("  " + "-"*32)

baseline_results = [
    ("CNN-BiLSTM+Optuna", 0.661),
]

best_mode_f1 = max(v["f1"] for v in results.values())

for mode, vals in results.items():
    marker = " ← best (ResNet)" if vals["f1"] == best_mode_f1 else ""
    print(f"  {mode:<15} | {vals['f1']:>6.4f} | "
          f"{vals['thresh']:>7.2f}{marker}")

print(f"\n  {'─'*32}")
print(f"  {'CNN-BiLSTM+Optuna':<15} | {'0.661':>6} | {'0.70':>7}  ← baseline")

winner_mode = max(results, key=lambda m: results[m]["f1"])
winner_f1   = results[winner_mode]["f1"]
diff        = winner_f1 - 0.661
arrow       = "↑" if diff > 0 else "↓"

print(f"\n  Best ResNet: {winner_mode} → F1={winner_f1:.4f} "
      f"{arrow} {abs(diff):.4f} vs baseline")

if winner_f1 > 0.661:
    print(f"\n  ✅ ResNet ({winner_mode}) beats CNN-BiLSTM+Optuna!")
    print(f"  Best model: {results[winner_mode]['path']}")
else:
    print(f"\n  CNN-BiLSTM+Optuna still the best at F1=0.661")

Mounted at /content/drive
✅ Ready
✅ Config | device: cuda
   Image size: 64×64
   Conversions: Spectrogram | GAF | Recurrence Plot
✅ Converters defined
✅ Dataset defined
✅ Model & helpers defined

Loading data...
   Train: (215858, 599) | Val: (623419, 599)

 MODE: SPECTROGRAM
  Epoch 01/20 | loss: 0.0116 | ✦ F1: 0.5827 (t=0.55) | lr: 9.9e-05 ← best 💾
  Epoch 02/20 | loss: 0.0061 | ✦ F1: 0.5418 (t=0.55) | lr: 9.8e-05
  Epoch 03/20 | loss: 0.0042 | lr: 9.5e-05
  Epoch 04/20 | loss: 0.0031 | ✦ F1: 0.6040 (t=0.55) | lr: 9.1e-05 ← best 💾
  Epoch 05/20 | loss: 0.0025 | lr: 8.6e-05
  Epoch 06/20 | loss: 0.0020 | ✦ F1: 0.5454 (t=0.45) | lr: 8.0e-05
  Epoch 07/20 | loss: 0.0017 | lr: 7.3e-05
  Epoch 08/20 | loss: 0.0014 | ✦ F1: 0.5593 (t=0.45) | lr: 6.6e-05
  Epoch 09/20 | loss: 0.0012 | lr: 5.8e-05
  Epoch 10/20 | loss: 0.0010 | ✦ F1: 0.5513 (t=0.50) | lr: 5.1e-05
  Epoch 11/20 | loss: 0.0009 | lr: 4.3e-05
  Epoch 12/20 | loss: 0.0007 | ✦ F1: 0.5506 (t=0.40) | lr: 3.5e-05
  Epoch 13/20 | loss

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")


# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, gc, pickle, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
warnings.filterwarnings("ignore")

BASE_PATH    = "/content/drive/MyDrive/new obada/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODEL_DIR    = BASE_PATH + "models/"
os.makedirs(MODEL_DIR, exist_ok=True)

APPLIANCE   = "dishwasher"
TRAIN_PKL   = PREPROCESSED + f"{APPLIANCE}_train.pkl"
VAL_PKL     = PREPROCESSED + f"{APPLIANCE}_val.pkl"
MODEL_PATH  = MODEL_DIR    + f"{APPLIANCE}_multiscale_best.pt"

WINDOW_SIZE  = 599
BATCH_SIZE   = 256
LR           = 8.99e-05
MAX_EPOCHS   = 40
PATIENCE     = 8
FULL_EVAL_EVERY = 2
RANDOM_SEED  = 42
FOCAL_GAMMA  = 2.97
FOCAL_ALPHA  = 0.578

# [AR_COMMENT_REMOVED]
BEST_PARAMS = {
    "f1_ch": 16, "f2_ch": 128, "f3_ch": 128,
    "hidden": 256, "n_layers": 1,
    "lstm_drop": 0.459, "clf_drop": 0.334,
    "weight_decay": 0.000567,
}

N_CHANNELS = 4   # raw + differential + scale5 + scale15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config | device: {device}")
print(f"   Input: {N_CHANNELS} channels (raw + diff + scale×5 + scale×15)")
print(f"   Architecture: Best Optuna CNN-BiLSTM")


# ============================================================
# CELL 3: Multi-scale Feature Extraction
# ============================================================
def build_multiscale(X, scale5=5, scale15=15):
    """
    Converts each window from (N, 599) to (N, 4, 599):

    Channel 2: slow scale (x5)  — 5-point moving average
    Channel 3: very slow scale (x15) — 15-point moving average
    """
    N, L = X.shape
    out  = np.zeros((N, 4, L), dtype=np.float32)

    # Channel 0: raw
    out[:, 0, :] = X

    # [AR_COMMENT_REMOVED]
    diff = np.diff(X, axis=1, prepend=X[:, :1])
    # [AR_COMMENT_REMOVED]
    d_max = np.abs(diff).max(axis=1, keepdims=True) + 1e-8
    out[:, 1, :] = diff / d_max

    # [AR_COMMENT_REMOVED]
    kernel5 = np.ones(scale5) / scale5
    for i in range(N):
        out[i, 2, :] = np.convolve(X[i], kernel5, mode='same')

    # [AR_COMMENT_REMOVED]
    kernel15 = np.ones(scale15) / scale15
    for i in range(N):
        out[i, 3, :] = np.convolve(X[i], kernel15, mode='same')

    return out


print("✅ Multi-scale builder defined")
print("   (Building takes ~2-3 min for 215K windows...)")


# ============================================================
# CELL 4: Dataset
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y_label):
        # X: (N, 4, 599) — already multi-scale
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y_label, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def load_pkl(path):
    with open(path, "rb") as f:
        d = pickle.load(f)
    return d["X"], d["y_label"]


def full_eval(model, X_ms, y_val, batch_size=512):
    ds     = NILMDataset(X_ms, np.zeros(len(X_ms), dtype=np.int8))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)
    probs  = []
    model.eval()
    with torch.no_grad():
        for X_b, _ in loader:
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(X_b.to(device))
            probs.append(torch.sigmoid(logits).cpu().numpy())
    probs  = np.concatenate(probs)
    labels = y_val.astype(int)
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.10, 0.99, 0.05):
        f1 = f1_score(labels, (probs >= t).astype(int),
                      pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return probs, best_f1, best_t


print("✅ Dataset & helpers defined")


# ============================================================
# CELL 5: Model — Multi-channel CNN-BiLSTM
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.97, alpha=0.578):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce   = nn.functional.binary_cross_entropy_with_logits(
                    logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t   = probs * targets + (1 - probs) * (1 - targets)
        a_t   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()


class MultiScaleCNNBiLSTM(nn.Module):
    """
    - First Conv takes 4 -> f1_ch (instead of 1 -> f1_ch)
    """
    def __init__(self, p=BEST_PARAMS, n_channels=N_CHANNELS):
        super().__init__()
        self.cnn = nn.Sequential(
            # Block 1: n_channels → f1_ch
            nn.Conv1d(n_channels, p["f1_ch"], kernel_size=15, padding=7),
            nn.BatchNorm1d(p["f1_ch"]), nn.ReLU(), nn.MaxPool1d(2),
            # Block 2
            nn.Conv1d(p["f1_ch"], p["f2_ch"], kernel_size=9, padding=4),
            nn.BatchNorm1d(p["f2_ch"]), nn.ReLU(), nn.MaxPool1d(2),
            # Block 3
            nn.Conv1d(p["f2_ch"], p["f3_ch"], kernel_size=5, padding=2),
            nn.BatchNorm1d(p["f3_ch"]), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.lstm = nn.LSTM(
            input_size=p["f3_ch"],
            hidden_size=p["hidden"],
            num_layers=p["n_layers"],
            batch_first=True,
            bidirectional=True,
            dropout=p["lstm_drop"] if p["n_layers"] > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Linear(p["hidden"] * 2, p["hidden"]),
            nn.ReLU(),
            nn.Dropout(p["clf_drop"]),
            nn.Linear(p["hidden"], 1),
        )

    def forward(self, x):
        # x: (B, 4, 599)
        x = self.cnn(x).permute(0, 2, 1)
        x, _ = self.lstm(x)
        return self.classifier(x[:, -1, :]).squeeze(1)


model = MultiScaleCNNBiLSTM().to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ MultiScale CNN-BiLSTM | params: {total:,}")
print(f"   Input channels: {N_CHANNELS} (raw + diff + scale5 + scale15)")


# ============================================================
# CELL 6: Build Multi-scale Data & Train
# ============================================================
print("\n" + "="*55)
print(" LOADING & BUILDING MULTI-SCALE DATA")
print("="*55)

X_train_raw, y_train = load_pkl(TRAIN_PKL)
X_val_raw, y_val_full = load_pkl(VAL_PKL)

print(f"   Raw train: {X_train_raw.shape}")
print(f"   Raw val  : {X_val_raw.shape}")

print("\n   Building train multi-scale... (215K windows)")
X_train_ms = build_multiscale(X_train_raw)
del X_train_raw
gc.collect()
print(f"   Train MS: {X_train_ms.shape}")

print("   Building val multi-scale... (623K windows)")
X_val_ms = build_multiscale(X_val_raw)
del X_val_raw
gc.collect()
print(f"   Val MS  : {X_val_ms.shape}")

train_ds     = NILMDataset(X_train_ms, y_train)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=LR,
                               weight_decay=BEST_PARAMS["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

best_f1      = 0.0
best_thresh  = 0.5
patience_cnt = 0

print(f"\n   Train batches: {len(train_loader)}")
print("\n" + "="*55)
print(" TRAINING")
print("="*55)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            loss = criterion(model(X_b), y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()
    lr_now = optimizer.param_groups[0]['lr']

    if epoch % FULL_EVAL_EVERY == 0 or epoch == 1:
        _, f1, best_t = full_eval(model, X_val_ms, y_val_full)

        flag = ""
        if f1 > best_f1:
            best_f1      = f1
            best_thresh  = best_t
            patience_cnt = 0
            torch.save({
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "best_f1":     best_f1,
                "best_thresh": best_thresh,
            }, MODEL_PATH)
            flag = " ← best 💾"
        else:
            patience_cnt += 1

        print(f"  Epoch {epoch:02d}/{MAX_EPOCHS} | loss: {train_loss:.4f} | "
              f"✦ F1: {f1:.4f} (t={best_t:.2f}) | lr: {lr_now:.1e}{flag}")
    else:
        print(f"  Epoch {epoch:02d}/{MAX_EPOCHS} | loss: {train_loss:.4f} | "
              f"lr: {lr_now:.1e}")

    if patience_cnt >= PATIENCE:
        print(f"\n  ⏹ Early stopping at epoch {epoch}")
        break

print(f"\n✅ Training complete")
print(f"   Best F1  : {best_f1:.4f}")
print(f"   Threshold: {best_thresh:.2f}")


# ============================================================
# CELL 7: Final Evaluation
# ============================================================
print("\n" + "="*55)
print(" FINAL EVALUATION (623K windows)")
print("="*55)

ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
print(f"   Loaded best model from epoch {ckpt['epoch']}")

probs_full, _, _ = full_eval(model, X_val_ms, y_val_full)
true_full = y_val_full.astype(int)

print(f"   Prob range: [{probs_full.min():.4f}, {probs_full.max():.4f}]")

print("\n" + "="*55)
print(" THRESHOLD SWEEP")
print("="*55)
print(f"  {'Thresh':>8} | {'F1':>6} | {'Precision':>10} | "
      f"{'Recall':>8} | {'TP':>7} | {'FP':>7} | {'FN':>7}")
print("  " + "-"*65)

best_thresh_f1, best_t_sw = 0.0, 0.5
for t in np.arange(0.10, 1.00, 0.05):
    preds = (probs_full >= t).astype(int)
    f1    = f1_score(true_full, preds, pos_label=1, zero_division=0)
    pr    = precision_score(true_full, preds, pos_label=1, zero_division=0)
    rc    = recall_score(true_full, preds, pos_label=1, zero_division=0)
    cm    = confusion_matrix(true_full, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (0,0,0,0)
    marker = " ←" if f1 > best_thresh_f1 else ""
    if f1 > best_thresh_f1:
        best_thresh_f1, best_t_sw = f1, t
    print(f"  {t:>8.2f} | {f1:>6.4f} | {pr:>10.4f} | {rc:>8.4f} | "
          f"{tp:>7,} | {fp:>7,} | {fn:>7,}{marker}")

print("\n" + "="*55)
print(f" BEST THRESHOLD: {best_t_sw:.2f}")
print("="*55)
best_preds = (probs_full >= best_t_sw).astype(int)
f1_fin = f1_score(true_full, best_preds, pos_label=1, zero_division=0)
pr_fin = precision_score(true_full, best_preds, pos_label=1, zero_division=0)
rc_fin = recall_score(true_full, best_preds, pos_label=1, zero_division=0)
cm_fin = confusion_matrix(true_full, best_preds)

print(f"   F1        : {f1_fin:.4f}")
print(f"   Precision : {pr_fin:.4f}")
print(f"   Recall    : {rc_fin:.4f}")
print(f"\n   Confusion Matrix:")
print(f"   {'':12} Pred OFF  Pred ON")
print(f"   {'True OFF':12} {cm_fin[0,0]:>8,}  {cm_fin[0,1]:>7,}")
print(f"   {'True ON':12} {cm_fin[1,0]:>8,}  {cm_fin[1,1]:>7,}")

print("\n" + "="*55)
print(" ALL APPROACHES — DISHWASHER SUMMARY")
print("="*55)
all_results = [
    ("CNN-BiLSTM + BCE",       0.560),
    ("CNN-BiLSTM + Focal",     0.574),
    ("Transformer",            0.466),
    ("CNN-BiLSTM + Optuna",    0.661),
    ("CtRNN + BiGRU",          0.178),
    ("ResNet + Spectrogram",   0.604),
    ("ResNet + GAF",           0.419),
    ("ResNet + Recurrence",    0.329),
    ("MultiScale CNN-BiLSTM",  f1_fin),
]
best_overall = max(r[1] for r in all_results)
print(f"  {'Model':<28} | {'F1':>6}")
print("  " + "-"*38)
for name, f1 in all_results:
    marker = " ✅ BEST" if f1 == best_overall else ""
    print(f"  {name:<28} | {f1:>6.4f}{marker}")

diff  = f1_fin - 0.661
arrow = "↑" if diff > 0 else "↓"
print(f"\n  MultiScale vs Optuna baseline: {arrow} {abs(diff):.4f}")

Mounted at /content/drive
✅ Drive mounted
✅ Config | device: cuda
   Input: 4 channels (raw + diff + scale×5 + scale×15)
   Architecture: Best Optuna CNN-BiLSTM
✅ Multi-scale builder defined
   (Building takes ~2-3 min for 215K windows...)
✅ Dataset & helpers defined
✅ MultiScale CNN-BiLSTM | params: 1,024,241
   Input channels: 4 (raw + diff + scale5 + scale15)

 LOADING & BUILDING MULTI-SCALE DATA
   Raw train: (215858, 599)
   Raw val  : (623419, 599)

   Building train multi-scale... (215K windows)
   Train MS: (215858, 4, 599)
   Building val multi-scale... (623K windows)
   Val MS  : (623419, 4, 599)

   Train batches: 844

 TRAINING
  Epoch 01/40 | loss: 0.0222 | ✦ F1: 0.6777 (t=0.75) | lr: 9.0e-05 ← best 💾
  Epoch 02/40 | loss: 0.0092 | ✦ F1: 0.6329 (t=0.60) | lr: 8.9e-05
  Epoch 03/40 | loss: 0.0070 | lr: 8.9e-05
  Epoch 04/40 | loss: 0.0056 | ✦ F1: 0.6844 (t=0.70) | lr: 8.8e-05 ← best 💾
  Epoch 05/40 | loss: 0.0048 | lr: 8.7e-05
  Epoch 06/40 | loss: 0.0042 | ✦ F1: 0.7011 (t=

In [ ]:
import os

models_dir = "/content/drive/MyDrive/new obada/models/"

# [AR_COMMENT_REMOVED]
to_delete = [
    "dishwasher_augmented_best.pt",      # augmentation → F1=0.385
    "dishwasher_best.pt",                # CNN-BiLSTM + BCE → F1=0.560
    "dishwasher_ctrnn_best.pt",          # CtRNN → F1=0.178
    "dishwasher_ms_optuna_best.pt",
    "dishwasher_optuna_best.pt",         # CNN-BiLSTM+Optuna → F1=0.661
    "dishwasher_resnet_gaf.pt",          # ResNet+GAF → F1=0.419
    "dishwasher_resnet_recurrence.pt",   # ResNet+Recurrence → F1=0.329
    "dishwasher_resnet_spectrogram.pt",  # ResNet+Spectrogram → F1=0.604
    "dishwasher_transformer_best.pt",    # Transformer → F1=0.466
]

for fname in to_delete:
    path = models_dir + fname
    if os.path.exists(path):
        os.remove(path)
        print(f"🗑️  Deleted: {fname}")
    else:
        print(f"⚠️  Not found: {fname}")

print("\n✅ Done! Remaining models:")
for f in sorted(os.listdir(models_dir)):
    size = os.path.getsize(models_dir + f) / 1e6
    print(f"   {f:<45} {size:.1f} MB")

🗑️  Deleted: dishwasher_augmented_best.pt
🗑️  Deleted: dishwasher_best.pt
🗑️  Deleted: dishwasher_ctrnn_best.pt
🗑️  Deleted: dishwasher_ms_optuna_best.pt
🗑️  Deleted: dishwasher_optuna_best.pt
🗑️  Deleted: dishwasher_resnet_gaf.pt
🗑️  Deleted: dishwasher_resnet_recurrence.pt
🗑️  Deleted: dishwasher_resnet_spectrogram.pt
🗑️  Deleted: dishwasher_transformer_best.pt

✅ Done! Remaining models:
   dishwasher_multiscale_best.pt                 4.1 MB
   fridge_v4_best.pt                             0.8 MB
   kettle_cnn_bilstm_best.pt                     8.9 MB
   kettle_v3_1to10_best.pt                       3.0 MB
   microwave_8sec_best.pth                       0.7 MB
   microwave_cnn_best.pth                        0.7 MB
   washing_machine_cnn_bilstm_best.pt            2.6 MB
